In [1]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import yfinance as yf

In [39]:
ticker = "HELI.CA"
data = yf.download(ticker, period="1y", auto_adjust=False, actions=True, progress=False)
print(data)

Price       Adj Close Capital Gains   Close Dividends    High     Low    Open  \
Ticker        HELI.CA       HELI.CA HELI.CA   HELI.CA HELI.CA HELI.CA HELI.CA   
Date                                                                            
2025-03-12  10.388981           0.0   10.80       0.0   11.03   10.78   10.90   
2025-03-13  10.369741           0.0   10.78       0.0   10.87   10.73   10.80   
2025-03-16  10.677564           0.0   11.10       0.0   11.11   10.81   10.78   
2025-03-17  10.485174           0.0   10.90       0.0   11.19   10.87   11.10   
2025-03-18  10.918049           0.0   11.35       0.0   11.35   10.84   10.90   
...               ...           ...     ...       ...     ...     ...     ...   
2026-03-08   5.200000           0.0    5.20       0.0    5.28    4.99    5.07   
2026-03-09   5.480000           0.0    5.48       0.0    5.49    5.21    5.20   
2026-03-10   5.640000           0.0    5.64       0.0    5.80    5.52    5.48   
2026-03-11   5.610000       

In [40]:
if getattr(data.columns, "nlevels", 1) > 1:
    high = data[("High", ticker)]
    split_events = data[("Stock Splits", ticker)] if ("Stock Splits", ticker) in data.columns else None
else:
    high = data["High"]
    split_events = data["Stock Splits"] if "Stock Splits" in data.columns else None

if split_events is None:
    split_factor = 1.0
else:
    split_factor = split_events.fillna(0).replace(0, 1).shift(-1, fill_value=1).iloc[::-1].cumprod().iloc[::-1]

split_adjusted_high = high / split_factor
high_52w_split_adjusted = split_adjusted_high.max()
high_52w_split_adjusted_date = split_adjusted_high.idxmax()

print(f"Split-adjusted 52W high for {ticker}: {high_52w_split_adjusted:.4f} on {high_52w_split_adjusted_date.date()}")
if split_events is not None and (split_events.fillna(0) != 0).any():
    print("Detected share-distribution events:")
    print(split_events[split_events.fillna(0) != 0])

Split-adjusted 52W high for HELI.CA: 5.8000 on 2026-03-10
Detected share-distribution events:
Date
2025-09-25    3.0
Name: (Stock Splits, HELI.CA), dtype: float64


In [41]:
ticker = "HELI.CA"
data = yf.download(ticker, period="1y", auto_adjust=False, actions=True, progress=False)

if getattr(data.columns, "nlevels", 1) > 1:
    high = data[("High", ticker)]
    close = data[("Close", ticker)]
    adj_close = data[("Adj Close", ticker)]
else:
    high = data["High"]
    close = data["Close"]
    adj_close = data["Adj Close"]

adj_factor = adj_close / close
adj_high = high * adj_factor

high_52w_adjusted = adj_high.max()
high_52w_adjusted_date = adj_high.idxmax()

print(f"Adjusted 52W High for {ticker}: {high_52w_adjusted:.4f} on {high_52w_adjusted_date.date()}")

Adjusted 52W High for HELI.CA: 11.4760 on 2025-06-01
